# Spotify Dataset Analysis — Tierra Canela 🇪🇨

Análisis exploratorio de audio features de Spotify + análisis de letras via **LRCLIB** con fallback a **lyrics.ovh**,
aplicado al grupo femenino de cumbia ecuatoriano **Tierra Canela** (fundado en 1998).

**Dataset:** [Spotify 1.2M+ Songs — Kaggle](https://www.kaggle.com/datasets/rodolfofigueroa/spotify-12m-songs)  
**Archivo necesario:** `tracks_features.csv`  
**Lyrics APIs:** https://lrclib.net/ + fallback https://lyrics.ovh/  
**Audio features:** https://developer.spotify.com/documentation/web-api/reference/get-audio-features

---


## 1. Setup & Carga de datos

In [ ]:
!pip install gdown wordcloud requests -q

import os, ast, time, warnings, requests, re
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud, STOPWORDS
from sklearn.manifold import TSNE, MDS
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
%matplotlib inline
sns.set_theme(style='whitegrid', palette='tab10')


In [ ]:
DATA_FILE = 'tracks_features.csv'

if not os.path.exists(DATA_FILE):
    print('Descargando dataset...')
    import gdown
    gdown.download(id='1jsXTNtGhOrsCApQctYx-hRxAQASAcPlI', output=DATA_FILE)
else:
    print(f'{DATA_FILE} ya existe, omitiendo descarga.')

df = pd.read_csv(DATA_FILE)
print(f'Dimensiones del dataset: {df.shape}')
df.head()


## 2. Vista general del dataset

In [ ]:
AUDIO_FEATURES = [
    'acousticness', 'danceability', 'duration_ms', 'energy',
    'instrumentalness', 'liveness', 'loudness', 'speechiness',
    'tempo', 'valence'
]

print('Valores nulos:')
print(df[AUDIO_FEATURES].isnull().sum())
print()
df[AUDIO_FEATURES].describe()


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
axes = axes.flatten()
for i, feat in enumerate(AUDIO_FEATURES):
    axes[i].hist(df[feat].dropna(), bins=50, color='#C0392B', edgecolor='none', alpha=0.75)
    axes[i].set_title(feat)
fig.suptitle('Distribuciones de Audio Features (Dataset Completo)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()


## 3. Filtrado por artista — Tierra Canela

Spotify Artist ID: `4UJR3YhhFBlHwUrav8WzPB`

| Campo | Valor |
|---|---|
| **Nombre** | Tierra Canela |
| **País** | Ecuador 🇪🇨 |
| **Género** | Cumbia / Tropical |
| **Fundación** | 1998 |
| **Sello** | La Herencia Musical Records |


In [ ]:
TARGET_ARTIST_ID = '4UJR3YhhFBlHwUrav8WzPB'  # Tierra Canela
ARTIST_NAME = 'Tierra Canela'

def artist_id_in_list(artist_ids_str: str, target_id: str) -> bool:
    try:
        return target_id in ast.literal_eval(artist_ids_str)
    except (ValueError, SyntaxError):
        return False

mask = df['artist_ids'].apply(lambda x: artist_id_in_list(str(x), TARGET_ARTIST_ID))
artist_df = df[mask].copy()
print(f'Filas encontradas para {ARTIST_NAME}: {len(artist_df)}')
artist_df[['name', 'album', 'release_date', 'year'] + AUDIO_FEATURES].head()


In [ ]:
# Diagnóstico: álbumes presentes en el dataset
artist_df['short_album_name'] = artist_df['album'].str.split('(').str[0].str.strip()
diag = (
    artist_df
    .groupby('short_album_name')
    .agg(tracks=('name', 'count'), year=('year', 'min'))
    .sort_values('year')
)
print('Álbumes encontrados en el dataset:')
print(diag.to_string())


### Álbumes de estudio confirmados

Se excluyen compilaciones (`Éxitos`, `Mix`), singles y colaboraciones.
Solo se conservan los álbumes de estudio publicados bajo La Herencia Musical Records.


In [ ]:
STUDIO_ALBUMS = {
    'Tu Recuerdo',           # 1998 — debut
    'Querido Ladrón',        # 2008
    'Llorando Tu Partida',   # 2012
    'Diosas de Cumbia',      # 2018/2021
}

artist_df = artist_df[artist_df['short_album_name'].isin(STUDIO_ALBUMS)]
artist_df = (
    artist_df
    .sort_values('year')
    .drop_duplicates(subset=['short_album_name', 'name'], keep='first')
    .sort_values('year')
    .reset_index(drop=True)
)

ALBUM_ORDER = (
    artist_df
    .drop_duplicates('short_album_name')
    .sort_values('year')['short_album_name']
    .tolist()
)

print(f'Canciones tras limpieza: {len(artist_df)}')
print('Álbumes (cronológico):', ALBUM_ORDER)

track_counts = artist_df.groupby('short_album_name')['name'].count().loc[ALBUM_ORDER]
track_counts.plot(kind='barh', figsize=(8, 4), color='#C0392B')
plt.xlabel('Número de canciones')
plt.title(f'Canciones por álbum de estudio — {ARTIST_NAME}')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


## 4. Scatter Plot: Acousticness vs. Valence

| Feature | Rango | Significado |
|---|---|---|
| **Acousticness** | 0–1 | Probabilidad de que el track sea acústico |
| **Valence** | 0–1 | Positividad musical (0=triste/oscuro, 1=alegre/eufórico) |

Tamaño de burbuja = duración de la canción.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 8))
sns.scatterplot(
    data=artist_df, x='valence', y='acousticness',
    hue='short_album_name', hue_order=ALBUM_ORDER,
    palette='tab10', size='duration_ms', sizes=(60, 900),
    alpha=0.75, ax=ax
)
handles, labels = ax.get_legend_handles_labels()
n = len(ALBUM_ORDER)
ax.legend(handles[1:n+1], labels[1:n+1], title='Álbum', loc='best')
ax.set_title(f'Acousticness vs. Valence por álbum — {ARTIST_NAME}', fontsize=13)
ax.set_xlabel('Valence (0=oscuro/triste · 1=alegre/eufórico)')
ax.set_ylabel('Acousticness (0=electrónico · 1=acústico)')
plt.tight_layout()
plt.show()


## 5. Radar Chart por álbum

Media de audio features por álbum, normalizados a [0, 1].

In [ ]:
RADAR_FEATURES = ['acousticness', 'danceability', 'energy',
                  'instrumentalness', 'liveness', 'valence']

album_means = artist_df.groupby('short_album_name')[RADAR_FEATURES].mean().loc[ALBUM_ORDER]
album_norm  = (album_means - album_means.min()) / (album_means.max() - album_means.min() + 1e-9)

N      = len(RADAR_FEATURES)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist() + [0]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
colors  = sns.color_palette('tab10', n_colors=len(album_norm))

for (album, row), color in zip(album_norm.iterrows(), colors):
    vals = row.tolist() + row.tolist()[:1]
    ax.plot(angles, vals, color=color, linewidth=1.8, label=album)
    ax.fill(angles, vals, color=color, alpha=0.10)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(RADAR_FEATURES, size=10)
ax.set_title(f'Audio Features medios por álbum (normalizados) — {ARTIST_NAME}', size=12, pad=22)
ax.legend(loc='upper right', bbox_to_anchor=(1.5, 1.15), fontsize=8)
plt.tight_layout()
plt.show()


## 6. Reducción de dimensionalidad

Features **estandarizadas (z-score)** antes de la proyección.

In [ ]:
REDUCTION_FEATURES = [
    'acousticness', 'danceability', 'duration_ms', 'energy',
    'instrumentalness', 'liveness', 'loudness', 'tempo', 'valence'
]

X  = artist_df[REDUCTION_FEATURES].dropna()
Xs = StandardScaler().fit_transform(X)
print(f'Matriz de features para proyección: {Xs.shape}')


In [ ]:
# t-SNE
tsne_coords = TSNE(
    n_components=2, perplexity=min(15, len(Xs) - 1), random_state=3
).fit_transform(Xs)
tsne_df = pd.DataFrame(tsne_coords, columns=['x', 'y'], index=X.index)
tsne_df['short_album_name'] = artist_df.loc[X.index, 'short_album_name'].values
tsne_df['duration_ms']      = artist_df.loc[X.index, 'duration_ms'].values

plt.figure(figsize=(9, 8))
ax = sns.scatterplot(
    data=tsne_df, x='x', y='y',
    hue='short_album_name', hue_order=ALBUM_ORDER,
    palette='tab10', size='duration_ms', sizes=(60, 700), alpha=0.8
)
handles, labels = ax.get_legend_handles_labels()
n = len(ALBUM_ORDER)
ax.legend(handles[1:n+1], labels[1:n+1], loc='best', ncol=2, title='Álbum')
plt.title(f't-SNE — Proyección de Audio Features ({ARTIST_NAME})')
plt.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# MDS
mds_coords = MDS(n_components=2, normalized_stress='auto', random_state=3).fit_transform(Xs)
mds_df = pd.DataFrame(mds_coords, columns=['x', 'y'], index=X.index)
mds_df['short_album_name'] = artist_df.loc[X.index, 'short_album_name'].values
mds_df['duration_ms']      = artist_df.loc[X.index, 'duration_ms'].values

plt.figure(figsize=(9, 8))
ax = sns.scatterplot(
    data=mds_df, x='x', y='y',
    hue='short_album_name', hue_order=ALBUM_ORDER,
    palette='tab10', size='duration_ms', sizes=(60, 700), alpha=0.8
)
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[1:n+1], labels[1:n+1], loc='best', ncol=2, title='Álbum')
plt.title(f'MDS — Proyección de Audio Features ({ARTIST_NAME})')
plt.axis('off')
plt.tight_layout()
plt.show()


## 7. Análisis de Letras — LRCLIB API + fallback lyrics.ovh

Siguiendo la lógica del `index.html`, primero se consulta **LRCLIB**:

`GET https://lrclib.net/api/search?track_name=X&artist_name=Tierra%20Canela`

Si LRCLIB no devuelve una letra utilizable, se usa **lyrics.ovh** como respaldo:

`GET https://api.lyrics.ovh/v1/Tierra%20Canela/X`

Se normalizan las letras sincronizadas eliminando timestamps tipo `[00:12.34]`, se cachean los resultados en memoria durante la ejecución y se omiten del análisis léxico las canciones sin letra disponible.


In [ ]:
LYRICS_USER_AGENT = 'TierraCanela-MusicNotebook/1.0 (academic-analysis)'
LYRICS_DELAY = 0.5
lyrics_cache = {}


def _clean_synced_lyrics(text: str) -> str:
    """Elimina timestamps LRC tipo [00:12.34] y normaliza saltos de línea."""
    if not text:
        return ''
    text = re.sub(r'\[\d{1,2}:\d{2}(?:\.\d{1,3})?\]\s*', '', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()


def fetch_lyrics_lrclib(artist: str, title: str, delay: float = LYRICS_DELAY):
    """Obtiene la letra desde LRCLIB. Devuelve str o None.

    LRCLIB puede devolver `plainLyrics` o `syncedLyrics`; si solo hay letras
    sincronizadas, se limpian los timestamps para poder analizarlas como texto.
    """
    url = 'https://lrclib.net/api/search'
    params = {'track_name': title, 'artist_name': artist}
    headers = {'User-Agent': LYRICS_USER_AGENT}

    try:
        resp = requests.get(url, params=params, headers=headers, timeout=10)
        time.sleep(delay)
        if resp.status_code != 200:
            return None

        results = resp.json()
        if not isinstance(results, list) or not results:
            return None

        # Prioriza coincidencias del artista y del título; si no, usa el primer resultado.
        title_norm = title.lower().strip()
        artist_norm = artist.lower().strip()
        best = None
        for item in results:
            item_track = str(item.get('trackName', '')).lower().strip()
            item_artist = str(item.get('artistName', '')).lower().strip()
            if title_norm in item_track and artist_norm in item_artist:
                best = item
                break
        if best is None:
            best = results[0]

        lyrics = best.get('plainLyrics') or best.get('syncedLyrics') or ''
        lyrics = _clean_synced_lyrics(lyrics)
        return lyrics if len(lyrics) > 20 else None
    except (requests.RequestException, ValueError, TypeError):
        return None


def fetch_lyrics_lyrics_ovh(artist: str, title: str, delay: float = LYRICS_DELAY):
    """Fallback compatible con la versión anterior del notebook: lyrics.ovh."""
    url = f'https://api.lyrics.ovh/v1/{requests.utils.quote(artist)}/{requests.utils.quote(title)}'
    try:
        resp = requests.get(url, timeout=10)
        time.sleep(delay)
        if resp.status_code == 200:
            lyrics = resp.json().get('lyrics', '')
            lyrics = _clean_synced_lyrics(lyrics)
            return lyrics if len(lyrics) > 20 else None
        return None
    except (requests.RequestException, ValueError, TypeError):
        return None


def fetch_lyrics(artist: str, title: str, delay: float = LYRICS_DELAY):
    """Obtiene letras usando LRCLIB primero y lyrics.ovh como respaldo.

    Devuelve un diccionario con la letra y la fuente para poder auditar la cobertura.
    """
    key = (artist.lower().strip(), title.lower().strip())
    if key in lyrics_cache:
        return lyrics_cache[key]

    lyrics = fetch_lyrics_lrclib(artist, title, delay=delay)
    source = 'LRCLIB' if lyrics else None

    if not lyrics:
        lyrics = fetch_lyrics_lyrics_ovh(artist, title, delay=delay)
        source = 'lyrics.ovh' if lyrics else None

    result = {'lyrics': lyrics or '', 'source': source or 'not_found'}
    lyrics_cache[key] = result
    return result


SPANISH_STOPWORDS = set(STOPWORDS) | {
    'que', 'de', 'el', 'la', 'los', 'las', 'un', 'una', 'y', 'a', 'en',
    'es', 'se', 'no', 'te', 'me', 'mi', 'si', 'por', 'con', 'para',
    'pero', 'más', 'lo', 'como', 'le', 'ya', 'su', 'al', 'del', 'hay',
    'yo', 'tú', 'tu', 'él', 'eso', 'ese', 'esta', 'este', 'o',
    'todo', 'toda', 'muy', 'tan', 'cuando', 'porque', 'soy', 'eres', 'fue',
    'ser', 'has', 'han', 'son', 'era', 'cada', 'qué', 'oh', 'ah', 'ay',
    'na', 'ooh', 'yeah', 'eh', 'hey', 'uh', 'mm', 'da', 'vas', 'viene',
    'voy', 'mis', 'sus', 'aquí', 'allí', 'bien', 'mal', 'ver', 'ir'
}


In [ ]:
print('Descargando letras con LRCLIB + fallback lyrics.ovh... (puede tardar unos minutos)')
lyrics_data = []

for _, row in artist_df.iterrows():
    album = row['short_album_name']
    track = row['name']
    result = fetch_lyrics(ARTIST_NAME, track)
    lyrics = result['lyrics']
    source = result['source']
    status = '✓' if lyrics else '✗'
    print(f'  [{status}] {album} — {track} ({source})')
    lyrics_data.append({
        'album': album,
        'track': track,
        'lyrics': lyrics,
        'lyrics_source': source,
    })

lyrics_df = pd.DataFrame(lyrics_data)
found = (lyrics_df['lyrics'] != '').sum()
print(f'\nLetras encontradas: {found}/{len(lyrics_df)}')
print('\nCobertura por fuente:')
print(lyrics_df['lyrics_source'].value_counts().to_string())


### 7a. Top 10 palabras más repetidas por álbum

In [ ]:
def top_words(texts, n=10, stopwords=SPANISH_STOPWORDS):
    words = []
    for t in texts:
        for w in t.lower().split():
            w_clean = ''.join(c for c in w if c.isalpha())
            if len(w_clean) > 2 and w_clean not in stopwords:
                words.append(w_clean)
    return Counter(words).most_common(n)

fig, axes = plt.subplots(1, len(ALBUM_ORDER), figsize=(5 * len(ALBUM_ORDER), 5), sharey=False)
if len(ALBUM_ORDER) == 1:
    axes = [axes]

palette = sns.color_palette('tab10', n_colors=len(ALBUM_ORDER))

for ax, album, color in zip(axes, ALBUM_ORDER, palette):
    texts = lyrics_df[lyrics_df['album'] == album]['lyrics'].tolist()
    words = top_words(texts, n=10)

    if not words:
        ax.text(0.5, 0.5, 'Sin letras\ndisponibles', ha='center', va='center',
                transform=ax.transAxes, fontsize=11, color='gray')
        ax.set_title(album, fontsize=10)
        ax.axis('off')
        continue

    labels, counts = zip(*words)
    y_pos = range(len(labels))
    ax.barh(y_pos, counts, color=color, alpha=0.85)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels, fontsize=9)
    ax.invert_yaxis()
    ax.set_title(album, fontsize=10, fontweight='bold')
    ax.set_xlabel('Frecuencia')

fig.suptitle(f'Top 10 palabras más repetidas por álbum — {ARTIST_NAME}', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()


### 7b. Nube de palabras global

In [ ]:
all_lyrics = ' '.join(lyrics_df['lyrics'].tolist())
word_freq  = {}
for word in all_lyrics.lower().split():
    clean = ''.join(c for c in word if c.isalpha())
    if len(clean) > 2 and clean not in SPANISH_STOPWORDS:
        word_freq[clean] = word_freq.get(clean, 0) + 1

if word_freq:
    wc = WordCloud(
        width=900, height=500,
        background_color='white',
        colormap='RdBu',
        max_words=120,
        prefer_horizontal=0.85,
    ).generate_from_frequencies(word_freq)

    plt.figure(figsize=(12, 6))
    plt.imshow(wc, interpolation='bilinear')
    plt.axis('off')
    plt.title(f'Nube de palabras global — {ARTIST_NAME}', fontsize=14, pad=12)
    plt.tight_layout()
    plt.show()
else:
    print('No hay letras suficientes para generar la nube de palabras.')


### 7c. Resumen: palabra #1 por álbum

In [ ]:
print('── Palabra más repetida por álbum ──')
for album in ALBUM_ORDER:
    texts = lyrics_df[lyrics_df['album'] == album]['lyrics'].tolist()
    words = top_words(texts, n=1)
    if words:
        word, count = words[0]
        print(f'  {album:30s}  →  "{word}" ({count} veces)')
    else:
        print(f'  {album:30s}  →  sin datos de letras')


## 8. Observaciones clave

- **Acousticness vs. Valence:** Tierra Canela tiende a valores de valence medio-altos (cumbia bailable), con acousticness variable según la época.
- **Radar chart:** compara la evolución sonora entre álbumes; los primeros discos suelen tener más energía bailable.
- **t-SNE / MDS:** los álbumes más cercanos entre sí comparten un perfil sonoro parecido; los outliers son canciones atípicas.
- **Letras:** la palabra más repetida refleja los temas centrales — amor romántico, añoranza e identidad femenina ecuatoriana. La nube de palabras global visualiza el vocabulario dominante en toda su discografía de estudio.

---
*Tip: cambia `TARGET_ARTIST_ID` y `STUDIO_ALBUMS` en la Sección 3 para explorar cualquier otro artista del dataset.*
